# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring (Lane 2, core lane).**

I'm picking this lane because it lines up with a question a real team actually has capacity to act on — which pages get reviewed first — and because I've already run the starter pipeline end to end for it, so I know the data supports it instead of guessing. The starter's own baseline-vs-model comparison (section 3 below) shows a learned ranking clearly beating a fixed rule on this exact task, which tells me there's real, non-trivial signal here worth spending seven weeks on. The lane also has a healthy base rate to learn from — over 40% of starter rows already show a "declining with demand" pattern — so I won't be fighting the sparse-label problem that the AI-referral freestyle direction warns about. My plan is to start from the starter's proxy label (`trend_direction == "down"`, calculated from the current window) to prove the workflow, then move to a proper future-window label (features from a prior window predicting an outcome in a later window) once I reach the modeling weeks, as the lane guide recommends for a stronger capstone.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Given a limited review budget, which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?

**Unit of analysis:** one content item (`content_id`) — one pseudonymized page, described by its trailing-90-day search and engagement metrics.

**Decision this improves:** which pages land in the "review this week" queue, out of far more candidates than a reviewer has time for.

**Who acts, and how:** a content reviewer / editor works down a ranked queue (highest score first) and, for each page, decides whether to refresh, expand, protect, prune, or just keep monitoring it — using the reason codes attached to each score to understand *why* it was flagged.

**Cost of a wrong call:**
- **False positive** (a page is flagged high-priority but doesn't actually need attention): wastes limited editor hours, and — because the reviewer's time is fixed — pushes a page that genuinely needed help further down the queue, where it may never get looked at.
- **False negative** (a genuinely declining or under-performing page never gets flagged): the traffic loss keeps compounding silently until someone notices by accident, by which point it's a bigger, slower fix.

Because both error types cost real time or real traffic, and because a reviewer only ever works through the *top* of the list, the metric that matches this decision is **precision@K** (K = however many pages the team can realistically act on), not raw accuracy across all 30,000 rows.

**Why data or ML helps at all:** a hand-written if/then rule (the starter's `baseline_refresh_score`) already catches some obvious cases — stale-but-visible pages, thin pages, low-CTR pages — but real pages fail for overlapping, differently-weighted reasons at once, and those weights likely shift across content types and clients. That is exactly the kind of messy, many-signal pattern a simple hand-tuned rule struggles with but a model can learn directly from evidence. The starter's own baseline-vs-model results below back this up.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/saadtalat111/flyrank"
REPO_DIR = "flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank
Starter data found. You're ready.


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}  |  Unique clients: {df['client_id'].nunique()}")
print()

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

print(f"declining_with_demand pages: {declining_with_demand:,} "
      f"({declining_with_demand / len(df):.1%} of all rows)")
print(f"low_ctr_visible_page pages:  {low_ctr_visible_page:,} "
      f"({low_ctr_visible_page / len(df):.1%} of all rows)")
print()
print("=> Both patterns are common enough (tens of percent of rows) to learn from safely —")
print("   unlike a sparse signal like AI-referral sessions.")

Rows: 30,000  |  Unique clients: 32

declining_with_demand pages: 13,152 (43.8% of all rows)
low_ctr_visible_page pages:  9,759 (32.5% of all rows)

=> Both patterns are common enough (tens of percent of rows) to learn from safely —
   unlike a sparse signal like AI-referral sessions.


In [5]:
with open("outputs/model_report.md") as f:
    report = f.read()

start = report.index("## Model Comparison")
end = report.index("## Final Queue")
print(report[start:end].strip())

## Model Comparison

Best model: `random_forest` selected by `precision_at_50`.

| Model | ROC AUC | Avg precision | Precision@50 | Recall | F1 |
|---|---:|---:|---:|---:|---:|
| decision_tree | 0.742 | 0.575 | 0.540 | 0.716 | 0.634 |
| logistic_regression | 0.700 | 0.522 | 0.400 | 0.567 | 0.566 |
| random_forest | 0.750 | 0.618 | 0.740 | 0.744 | 0.640 |
| baseline_rules | 0.627 | 0.468 | 0.240 | - | - |


**Reading these numbers:** on this 30,000-row starter slice, a fixed rule (`baseline_rules`) gets about 12 of its top 50 flagged pages right (precision@50 = 0.240). A random forest trained on the same observable signals gets about 37 of its top 50 right (precision@50 = 0.740) — using client-holdout validation, so it was tested on clients it never saw in training. That's a large, honest gap on the exact metric (precision@K) that matches how a reviewer would actually use the queue, and it's the strongest evidence that this lane is worth building out further over the next 7 weeks.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- This is **decision-support**: a ranked list that helps a reviewer spend limited time on the most promising pages first — not a guarantee that fixing a flagged page will work.
- Results are **observational**: I can say a page's signals are *associated with* decline or opportunity, and I can measure whether my ranking beats a transparent baseline on a held-out set of clients.
- Claims will be phrased as "observed," "measured," or "this suggests" — never "proven."

**What I will never claim:**
- That refreshing a page **caused** a recovery — that needs a real experiment (e.g. an A/B test), which this data can't give me.
- Anything about **Google's ranking algorithm**, or that I "predicted" a Google ranking factor.
- Anything about **AI citations or AI search rankings** — the AI-related columns only measure sessions where someone clicked through from an AI tool, nothing about whether/how an AI model cited the content.
- That the starter's `is_declining_label` (`trend_direction == "down"`) is the "true" definition of decline — it's a simple, current-window proxy I'm using to prove the workflow early; a stronger, future-window label (prior 90 days of features → next 30 days outcome) is the target for later weeks.
- That a flagged page is guaranteed to need help, or that an unflagged page is guaranteed to be fine — every recommendation still needs a human reviewer's judgment.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.